# T0.5 · Data extraction

Prepares `matrix.csv` with one row per included study, summarises the pre-coding made by AI from the abstracts and tracks the progress of the extraction.

- **Input:** `corpus.csv` of T0.2, the decisions of T0.3, the form in `../T0.1-protocol/extraction-form.md` and `precoding_ai.csv`.
- **Output:** `matrix.csv`, which is filled in by hand from the full text. The identification columns are set by this notebook.
- **To re-run:** run everything. What is already written in `matrix.csv` is kept; studies that become included are added as empty rows.

In [1]:
import re, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", 100)
T02 = Path("../T0.2-search-and-deduplication")
T03 = Path("../T0.3-screening")
MATRIX = Path("matrix.csv")
PRECODING = Path("precoding_ai.csv")

YPN, YN = ["y", "p", "n"], ["y", "n"]
VALUES = {      # coded columns of the form and their valid values
    "level": ["ic", "board", "both", "ns"],
    "stage": ["production", "maintenance", "field", "design", "ns"],
    "validation": ["simulation", "hardware", "both"],
    "sel_node": YN, "sel_stimulus": YN, "sel_measurement": YN,
    "coupling": ["joint", "sequential", "single"],
    "fault_type": ["catastrophic", "parametric", "both"],
    "multiple_faults": YN, "multiobjective": YN,
    "weight_probability": YPN, "weight_severity": YPN, "false_alarms": YPN,
    "cost": ["none", "count", "abstract", "time", "hardware"],
    "instrument_model": YPN, "accessibility": YN,
    "tolerances": ["ignored", "threshold", "parametric", "montecarlo"],
    "mc_distribution": ["normal", "uniform", "other", "ns"],
    "intervals": YN,
    "family": ["exhaustive", "greedy", "graph", "metaheuristic", "exact", "ml", "rl", "symbolic", "other"],
    "guarantee": YN, "cmp_optimum": YN, "cmp_bound": YN,
    "rep_netlist": YPN, "rep_faults": YPN, "rep_parameters": YPN,
    "C1_joint": YPN, "C2_hw_cost": YPN, "C3_risk": YPN, "C4_certification": YPN,
    "C5a_reference": YPN, "C5b_openness": YPN,
}
GAP = ["C1_joint", "C2_hw_cost", "C3_risk", "C4_certification", "C5a_reference", "C5b_openness"]
COLUMNS = (
    ["id", "citation", "doi", "group", "level", "stage", "validation", "circuits", "n_components",
     "sel_node", "sel_stimulus", "sel_measurement", "stimulus_type", "coupling", "n_candidates",
     "fault_type", "multiple_faults", "n_faults",
     "objective", "multiobjective", "weight_probability", "weight_severity", "false_alarms",
     "cost", "instrument_model", "accessibility",
     "tolerances", "mc_runs", "mc_distribution", "intervals",
     "family", "guarantee", "cmp_optimum", "cmp_bound", "baselines",
     "rep_netlist", "rep_faults", "rep_parameters", "code", "data"]
    + [c for g in GAP for c in (g, g + "_quote")] + ["g0_candidate", "notes"]
)

corpus = pd.read_csv(T02 / "corpus.csv", dtype=str).fillna("").set_index("id")
ai = pd.read_csv(T03 / "screening_ai.csv", dtype=str).fillna("").set_index("id")["decision"]
human = pd.read_csv(T03 / "screening.csv", dtype=str).fillna("").set_index("id")["decision"]
corpus["decision"] = human.where(human != "", ai).reindex(corpus.index).fillna("")
included = corpus[corpus["decision"] == "include"]
print("Studies included according to the screening decision in force:", len(included))
print("  with a human decision:", int((human.reindex(included.index) != "").sum()))

Studies included according to the screening decision in force: 468
  with a human decision: 0


## 1. Extraction matrix

One row per included study. As long as the full-text screening is not done, the list of included studies is provisional.

In [2]:
def citation(row):
    first = re.split(r"[;,]", row["authors"])[0].strip()
    return f"{first} ({row['year']}), {row['source']}"[:140]

new = pd.DataFrame("", index=included.index, columns=COLUMNS[1:])
new["citation"] = included.apply(citation, axis=1)
new["doi"] = included["doi"]

if MATRIX.exists():
    previous = pd.read_csv(MATRIX, dtype=str).fillna("").set_index("id")
    common = new.index.intersection(previous.index)
    manual = [c for c in previous.columns if c in new.columns and c not in ("citation", "doi")]
    new.loc[common, manual] = previous.loc[common, manual]
    outside = previous.index.difference(new.index)
    with_data = [i for i in outside if (previous.loc[i, manual] != "").any()]
    if with_data:
        print("Studies with extracted data that are no longer listed as included (kept):", ", ".join(with_data))
        new = pd.concat([new, previous.loc[with_data].reindex(columns=new.columns, fill_value="")])

matrix = new.sort_index()
matrix.to_csv(MATRIX, index_label="id", encoding="utf-8")
print("Rows in matrix.csv:", len(matrix))

Rows in matrix.csv: 468


## 2. Pre-coding (AI)

`precoding_ai.csv` is a first approximation made by an AI assistant **from the abstract**, not from the full text. It covers only what an abstract lets one see and uses simplified codes:

- `tolerances`: `y` if the abstract says it considers tolerances or process variation; `ns` if it does not say.
- `hw_validation`: `y` if the abstract mentions measurements or validation in hardware; `ns` otherwise.
- `family` also admits `sensitivity` (analytical methods based on sensitivities).
- `C2_hw_cost`, `C3_risk` and `C4_certification` follow the definitions of the protocol, with `p` when the abstract only hints at it.

**Limitation:** the beginning and the end of each abstract were read, and an abstract seldom says how many Monte Carlo runs were used or which instrument is modelled. An `n` here means "the abstract does not mention it", not "the paper does not do it".

In [3]:
pre = pd.read_csv(PRECODING, dtype=str).fillna("").set_index("id")
pre = pre.join(corpus[["year", "title"]])
not_precoded = included.index.difference(pre.index)
print(f"Pre-coded studies: {len(pre)} · included but not pre-coded: {len(not_precoded)}")

def breakdown(col, order=None):
    s = pre[col].value_counts()
    return s.reindex(order).fillna(0).astype(int) if order else s

pd.DataFrame({
    "level": breakdown("level"), "stage": breakdown("stage"), "objective": breakdown("objective"),
}).fillna(0).astype(int).replace(0, "")

Pre-coded studies: 468 · included but not pre-coded: 0


,level,stage,objective
board,33,,
design,,24,
detection,,,153
field,,22,
ic,148,,
identification,,,76
isolation,,,163
maintenance,,27,
ns,287,235,
production,,160,


What is selected (RQ1). "Decisions" counts how many of the three things each study chooses: node, stimulus and measurement.

In [4]:
sel = pre[["sel_node", "sel_stimulus", "sel_measurement"]].eq("y")
pre["decisions"] = sel.sum(axis=1)
combination = sel.apply(lambda r: " + ".join(n for n, v in zip(["node", "stimulus", "measurement"], r) if v) or "none (testability)", axis=1)
print(pre["decisions"].value_counts().sort_index().rename("studies by number of decisions").to_string(), "\n")
combination.value_counts().rename("studies").to_frame()

decisions
0     55
1    375
2     32
3      6 



,studies
stimulus,192
node,98
measurement,85
none (testability),55
node + stimulus,20
stimulus + measurement,10
node + stimulus + measurement,6
node + measurement,2


Algorithm, tolerances and validation (RQ4 and RQ5).

In [5]:
print("Considers tolerances or process variation (according to the abstract):", int((pre["tolerances"] == "y").sum()))
print("Mentions validation in hardware:", int((pre["hw_validation"] == "y").sum()), "\n")
breakdown("family").rename("studies").to_frame()

Considers tolerances or process variation (according to the abstract): 82
Mentions validation in hardware: 33 



,studies
family,
other,256
metaheuristic,57
sensitivity,38
ml,33
greedy,23
exact,19
graph,18
symbolic,15
exhaustive,7


### Gap criteria and G0 candidates

Studies by how many of the three decisive criteria (C2 hardware cost, C3 risk, C4 certification) appear in the abstract as `y` or `p`, split by level.

In [6]:
C = ["C2_hw_cost", "C3_risk", "C4_certification"]
pre["n_criteria"] = pre[C].ne("n").sum(axis=1)
print(pre[C].apply(lambda s: s.value_counts()).reindex(YPN).fillna(0).astype(int).to_string(), "\n")
pd.crosstab(pre["n_criteria"].rename("criteria present"), pre["level"], margins=True)

   C2_hw_cost  C3_risk  C4_certification
y           3       14                 1
p          99       37                18
n         366      417               449 



level,board,ic,ns,All
criteria present,,,,
0,8,61,268,337
1,25,51,17,93
2,0,33,2,35
3,0,3,0,3
All,33,148,287,468


In [7]:
candidates = pre[pre["n_criteria"] == 3].sort_values("year")
print("Studies with all three criteria (G0 candidates according to the abstract):", len(candidates))
candidates[["year", "level", "stage"] + C + ["notes", "title"]]

Studies with all three criteria (G0 candidates according to the abstract): 3


,year,level,stage,C2_hw_cost,C3_risk,C4_certification,notes,title
id,,,,,,,,
R1049,2008,ic,production,p,p,p,estimates the confidence of each prediction when compacting tests,Confidence estimation in non-RF to RF correlation-based specification test compaction
R1436,2014,ic,production,p,y,p,minimises specification tests without exceeding a defect level; Monte Carlo,Specification test minimization for given defect level
R1582,2018,ic,production,p,y,p,adaptive test with escape estimation,Adaptive Test with Test Escape Estimation for Mixed-Signal ICs


Board-level studies: which criteria they hint at.

In [8]:
board = pre[pre["level"] == "board"]
print("Board-level studies:", len(board))
print(board.groupby(C).size().rename("studies").to_string(), "\n")
board[board["n_criteria"] >= 1].sort_values("year")[["year", "stage"] + C + ["notes", "title"]].head(40)

Board-level studies: 33
C2_hw_cost  C3_risk  C4_certification
n           n        n                    8
            p        n                    2
p           n        n                   23 



,year,stage,C2_hw_cost,C3_risk,C4_certification,notes,title
id,,,,,,,
R0105,1988,production,p,n,n,optimal settings of the in-circuit test equipment,Parameter design solutions for analog in-circuit testing problems.
R0356,1997,maintenance,p,n,n,limited node access on unpowered boards,Analog fault diagnosis for unpowered circuit boards
R0520,1999,field,p,n,n,test under the control of a microcontroller,Improved procedures for testing analog and mixed-signal systems using a microcontroller
R0605,2000,maintenance,p,n,n,limited physical access on boards,Test point selection for analog fault diagnosis of unpowered circuit boards
R0886,2005,field,p,n,n,self-test with the microcontroller's resources; with tolerances,Implementation of an input-output method of diagnosis of analog electronic circuits in embedded ...
R0849,2005,field,p,n,n,self-test with the microcontroller's resources,About fault diagnosis methods of analog electronic circuits based on the time response to a squa...
R0906,2006,field,p,n,n,self-test with the microcontroller's resources,A fault diagnosis method of analog electronic circuits for mixed-signal systems controlled by mi...
R0905,2006,field,p,n,n,self-test with the microcontroller's resources,A fault diagnosis method of analog electronic circuits for mixed-signal systems controlled by mi...
R1027,2007,field,p,n,n,self-test with the microcontroller's resources,Using a square-wave signal for fault diagnosis of analog parts of mixed-signal embedded systems ...


Studies with two of the three criteria: these are the closest comparators.

In [9]:
two = pre[pre["n_criteria"] == 2].sort_values(["level", "year"])
print("Studies with two criteria:", len(two))
two[["year", "level"] + C + ["notes", "title"]]

Studies with two criteria: 35


,year,level,C2_hw_cost,C3_risk,C4_certification,notes,title
id,,,,,,,
R0466,1998,ic,n,y,p,Bayes' rule with prior probabilities to choose measurements,Probabilistic fault detection and the selection of measurements for analog integrated circuits
R0534,1999,ic,p,n,p,accounts for measurement inaccuracy; Monte Carlo,Speed-up of high accuracy analog test stimulus optimization
R0535,1999,ic,p,n,p,accounts for measurement inaccuracy; Monte Carlo,Speed-up of high accurate analog test stimulus optimization
R1950,2006,ic,p,y,n,minimises test cost while limiting yield loss,Statistical test compaction using binary decision trees
R0993,2007,ic,n,y,p,compaction with a statistical performance model (Monte Carlo),Minimization of functional tests by statistical modelling of analogue circuits
R0980,2007,ic,n,y,p,evaluates test measurements with a multivariate normal statistical model,Evaluation of test measures for LNA production testing using a multinormal statistical model
R1957,2007,ic,p,p,n,,Non-RF to RF test correlation using learning machines: A case study
R1086,2008,ic,p,p,n,adaptive test: test time versus quality,Test cost minimization through adaptive test development
R1973,2009,ic,p,p,n,per-device adaptive test,Adaptive test elimination for analog/RF circuits


## 3. Extraction progress

Progress of `matrix.csv`, validity of the codes and agreement with the pre-coding on the shared columns.

In [10]:
m = pd.read_csv(MATRIX, dtype=str).fillna("").set_index("id")
coded = [c for c in VALUES if c in m.columns]
started = (m[coded] != "").any(axis=1)
complete = (m[coded] != "").all(axis=1)
print(f"Rows started: {int(started.sum())} of {len(m)} · complete: {int(complete.sum())}")

errors = [(i, c, m.at[i, c]) for c in coded for i in m.index
          if m.at[i, c] and any(v.strip() not in VALUES[c] for v in m.at[i, c].split(";"))]
if errors:
    display(pd.DataFrame(errors, columns=["id", "column", "invalid value"]))

no_quote = [(i, g) for g in GAP for i in m.index if m.at[i, g] in ("y", "p") and not m.at[i, g + "_quote"]]
if no_quote:
    print("Gap criteria coded y or p without a literal quote:", len(no_quote))

if started.any():
    shared = ["level", "stage", "sel_node", "sel_stimulus", "sel_measurement", "C2_hw_cost", "C3_risk", "C4_certification"]
    both = m[started].join(pre[shared], rsuffix="_ai", how="inner")
    agreement = {c: (both[c] == both[c + "_ai"])[both[c] != ""].mean() for c in shared}
    display(pd.Series(agreement).map("{:.0%}".format).rename("agreement with the pre-coding").to_frame())

Rows started: 0 of 468 · complete: 0


## Conclusions and notes

Pre-coding (AI) of the 468 studies that the first pass of the screening takes as included, made from the abstracts only. `matrix.csv` is created and empty: full-text extraction has not started.

- **Evidence in favour of G0.** No abstract brings together hardware cost, risk and certification at board level. The only three studies where all three criteria appear, and always partially, are about production test of integrated circuits: R1049 (confidence of the prediction when compacting tests), R1436 (minimum set of specification tests for a given defect level) and R1582 (adaptive test with escape estimation).
- **The gap has the expected shape.** The 33 board-level studies hint at one criterion at most: 23 at hardware cost or resources (nearly all of them, self-test with a microcontroller), 2 at risk and none at certification. Risk and statistics appear in integrated circuits: the 35 studies with two criteria belong to that field except two with no stated level.
- **Joint selection (RQ1).** 375 studies choose a single thing (192 the stimulus, 98 the node, 85 the measurement), 38 choose two or three and 55 only analyse testability. The six that choose node, stimulus and measurement are the tool by Zilch and co-authors and the work by Chakrabarti and Chatterjee.
- **Tolerances and validation.** 82 abstracts say they consider tolerances or process variation and 33 mention hardware. Only one (R0477, 1998) asks how many Monte Carlo runs are needed.
- **What this is worth.** It guides the reading; it does not decide G0: an abstract does not give the number of runs or the instrument model, so C2 and C4 are underestimated, and `level` is not stated in 287 studies. The decision needs the full text of, at least, the 38 studies with two or three criteria and the 33 board-level ones.
- **Provisional.** The list of included studies comes from the first pass of the screening; it will change with the human review and with the backward citations pending in T0.4.

### Notes